# Notebook 04 — SHAP Explainability for the Final XGBoost Model

**Project:** Telco Customer Churn Analysis

This notebook explains **why the final XGBoost model** (selected in Notebook 03)
makes the predictions it makes, using **SHAP (SHapley Additive exPlanations)**
values computed from the *actual fitted model* and *actual customer features*.

It provides both:

1. **Global SHAP analysis** — which transformed features, on average, move the
   model's output the most across the training data.
2. **Per-customer local SHAP explanation** — for a single customer, how each
   feature contributed to that customer's specific predicted churn probability.

**Grounding principle:** every SHAP value, feature ranking, prediction
probability, and local contribution in this notebook is computed at run time
from the reproduced XGBoost model and `data/cleaned/telco_churn_clean.csv`.
Nothing is invented or hard-coded.

**Relationship to earlier notebooks:**

- Notebook 02 — statistical association in the observed dataset.
- Notebook 03 — predictive performance on unseen (holdout) data.
- Notebook 04 — **model attribution**: SHAP describes the behaviour of the
  fitted predictive model, *not* real-world causality.

**Output for Notebook 05:** the local explanation function produces a
structured "evidence" object per customer that Notebook 05 will consume when
generating natural-language explanations. No LLM is used in this notebook.

## 2. Imports

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost
import shap

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Load and Validate the Cleaned Dataset

Identical loading and validation to Notebook 03, with a path robust to the
kernel starting from the repository root or the `notebooks/` directory.

In [2]:
DATA_FILE = "data/cleaned/telco_churn_clean.csv"


def find_data_path() -> Path:
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidate = folder / DATA_FILE
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate {DATA_FILE}. Run this notebook from the repository "
        "root or from the notebooks/ directory."
    )


DATA_PATH = find_data_path()
PROJECT_ROOT = DATA_PATH.parents[2]  # repository root
print("Resolved data path:", DATA_PATH)

EXPECTED_ROWS = 7032

df = pd.read_csv(DATA_PATH)
print("Loaded shape:", df.shape)

problems = []
if df.shape[0] != EXPECTED_ROWS:
    problems.append(f"Expected {EXPECTED_ROWS} rows, found {df.shape[0]}")
if "Churn" not in df.columns:
    problems.append("Churn column is missing")
elif not set(df["Churn"].unique()).issubset({0, 1}):
    problems.append(f"Churn contains unexpected values: {sorted(df['Churn'].unique())}")
if problems:
    raise ValueError("Dataset validation failed:\n- " + "\n- ".join(problems))

missing = df.isna().sum()
print("Missing values per column:")
print(missing[missing > 0].to_string() if (missing > 0).any() else "  None")
print("Duplicate rows (exact, keep='first'):", int(df.duplicated().sum()))
print(f"Overall churn rate: {(df['Churn'] == 1).mean():.4f}")

Resolved data path: /Users/vaibhavvikasranjan/Downloads/telco-churn-analysis/data/cleaned/telco_churn_clean.csv
Loaded shape: (7032, 20)
Missing values per column:
  None
Duplicate rows (exact, keep='first'): 22
Overall churn rate: 0.2658


## 4. Load the Persisted Model Artifact

Notebook 03 persists the complete fitted **preprocessing + XGBoost pipeline** as
a versioned artifact. This notebook **loads** that artifact instead of
retraining the model, eliminating the possibility of preprocessing or model
drift between modeling and explainability.

- Model artifact: `models/telco_churn_xgboost.joblib`
- Metadata: `models/telco_churn_xgboost_metadata.json`
- SHA-256 checksum: `models/telco_churn_xgboost.joblib.sha256`

The modeling data is re-split exactly as Notebook 03 did (same `random_state`
and fractions) only to provide the training set as the SHAP explanation dataset
and the holdout set for a **model-identity check**. No model is fitted and no
hyperparameters are tuned in this notebook. The decision threshold is read from
the artifact metadata (0.525) rather than duplicated as a magic number.


In [3]:
df_model = df.drop_duplicates(keep="first")
X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)
print("Train / Dev / Holdout:", len(y_train), "/", len(y_dev), "/", len(y_holdout))
print("Modeling set rows:", len(df_model))

Train / Dev / Holdout: 4907 / 1051 / 1052
Modeling set rows: 7010


In [4]:
import hashlib
import json
import joblib

MODEL_DIR = DATA_PATH.parents[2] / "models"
MODEL_PATH = MODEL_DIR / "telco_churn_xgboost.joblib"
METADATA_PATH = MODEL_DIR / "telco_churn_xgboost_metadata.json"
HASH_PATH = MODEL_DIR / "telco_churn_xgboost.joblib.sha256"

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model artifact not found at {MODEL_PATH}. Run Notebook 03 first to "
        "persist the final pipeline."
    )


def compute_file_hash(path, algorithm="sha256"):
    """Return the hex digest of a file, streaming to bound memory use."""
    digest = hashlib.new(algorithm)
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


with open(METADATA_PATH) as f:
    artifact_metadata = json.load(f)

stored_hash = HASH_PATH.read_text().strip()
current_hash = compute_file_hash(MODEL_PATH)
print("Stored SHA-256 :", stored_hash)
print("Current SHA-256:", current_hash)
assert stored_hash == current_hash, "Artifact hash mismatch: file may have been replaced."
assert stored_hash == artifact_metadata["sha256"], "Hash does not match metadata sha256."

final_pipeline = joblib.load(MODEL_PATH)
print("Loaded pipeline steps:", list(final_pipeline.named_steps.keys()))
print("Metadata model type:", artifact_metadata["model_type"])

fitted_preprocessor = final_pipeline.named_steps["preprocess"]
fitted_xgb = final_pipeline.named_steps["classifier"]
assert isinstance(fitted_preprocessor, ColumnTransformer), "Unexpected preprocessor type"
assert isinstance(fitted_xgb, XGBClassifier), "Unexpected estimator type"

DECISION_THRESHOLD = float(artifact_metadata["decision_threshold"])
print("Decision threshold loaded from metadata:", DECISION_THRESHOLD)
print("No model was fitted in this notebook.")


Stored SHA-256 : 79cfdceeda66e597ed406f478c1d25cd28da242294bc58e87da3706bbfb9ae8f
Current SHA-256: 79cfdceeda66e597ed406f478c1d25cd28da242294bc58e87da3706bbfb9ae8f
Loaded pipeline steps: ['preprocess', 'classifier']
Metadata model type: Pipeline(ColumnTransformer preprocessing + XGBClassifier)
Decision threshold loaded from metadata: 0.525
No model was fitted in this notebook.


In [5]:
# Model identity verification: the loaded artifact must reproduce the exact
# Notebook 03 holdout results. This is an integrity check only; it feeds no
# information back into any decision.
proba_holdout = fitted_xgb.predict_proba(
    fitted_preprocessor.transform(X_holdout)
)[:, 1]
pred_holdout = (proba_holdout >= DECISION_THRESHOLD).astype(int)

loaded_roc = roc_auc_score(y_holdout, proba_holdout)
loaded_f1 = f1_score(y_holdout, pred_holdout)
loaded_cm = confusion_matrix(y_holdout, pred_holdout).ravel()

meta_eval = artifact_metadata["holdout_evaluation"]
print("Loaded-artifact holdout verification (integrity check only):")
print("  ROC-AUC:", round(loaded_roc, 4), "| metadata:", round(meta_eval["roc_auc"], 4))
print("  F1     :", round(loaded_f1, 4), "| metadata:", round(meta_eval["f1"], 4))
print("  Confusion matrix:", list(loaded_cm), "| metadata:", meta_eval["confusion_matrix"])

assert np.isclose(loaded_roc, meta_eval["roc_auc"], atol=1e-4)
assert np.isclose(loaded_f1, meta_eval["f1"], atol=1e-4)
assert list(loaded_cm) == [int(x) for x in meta_eval["confusion_matrix"]]

# Cross-check against the documented Notebook 03 holdout values.
assert abs(loaded_roc - 0.8530) < 1e-4
assert abs(loaded_f1 - 0.6365) < 1e-4
assert list(loaded_cm) == [585, 188, 61, 218]

print("Model identity verification PASSED: the persisted artifact is the exact "
      "model evaluated in Notebook 03. Execution continues.")


Loaded-artifact holdout verification (integrity check only):
  ROC-AUC: 0.853 | metadata: 0.853
  F1     : 0.6365 | metadata: 0.6365
  Confusion matrix: [np.int64(585), np.int64(188), np.int64(61), np.int64(218)] | metadata: [585, 188, 61, 218]
Model identity verification PASSED: the persisted artifact is the exact model evaluated in Notebook 03. Execution continues.


## 5. Transformed Features and Feature Names

The XGBoost model never sees the raw 19 features: it sees the output of the
fitted preprocessing pipeline (4 scaled numerical features plus the one-hot
encoded categorical columns). The **actual** transformed feature names are
extracted from the fitted `ColumnTransformer` with `get_feature_names_out()`
so that SHAP values align exactly with the model's feature columns — nothing is
invented.

In [6]:
X_train_transformed = fitted_preprocessor.transform(X_train)
feature_names = fitted_preprocessor.get_feature_names_out()
X_train_transformed = pd.DataFrame(X_train_transformed, columns=feature_names)

print("Transformed feature matrix shape:", X_train_transformed.shape)
print("Number of transformed features:", X_train_transformed.shape[1])
print("First 10 feature names:", list(feature_names[:10]))

assert X_train_transformed.shape[1] == len(feature_names)
assert X_train_transformed.shape[1] == fitted_xgb.get_booster().num_features()
print("Assertions passed: feature names align with the fitted model's feature count.")

Transformed feature matrix shape: (4907, 45)
Number of transformed features: 45
First 10 feature names: ['numerical__SeniorCitizen', 'numerical__tenure', 'numerical__MonthlyCharges', 'numerical__TotalCharges', 'categorical__gender_Female', 'categorical__gender_Male', 'categorical__Partner_No', 'categorical__Partner_Yes', 'categorical__Dependents_No', 'categorical__Dependents_Yes']
Assertions passed: feature names align with the fitted model's feature count.


## 6. SHAP Explainer

**What SHAP is.** SHAP values attribute the model's output for an observation
to its features. Each feature gets a number: how much that feature pushed the
output away from a base value.

**Three distinct quantities must not be confused:**

- **Model probability** — `predict_proba`, the chance of churn in [0, 1].
- **Model raw output / margin** — XGBoost's raw log-odds output *before* the
  sigmoid. `sigmoid(margin) = probability`.
- **SHAP contribution** — the share of the margin attributed to each feature.
  They are on the **raw-output (log-odds) scale**, not the probability scale.

For the binary XGBoost model, `shap.TreeExplainer` computes the positive-class
churn output. Because the model is a tree ensemble, the explainer uses
`feature_perturbation="tree_path_dependent"` (the supported, fast path for
XGBoost in this SHAP version), and **no background dataset is required**.

**Additivity (verified in section 16):** for every observation,
`sum(SHAP values) + base_value = margin`, exactly, because tree SHAP is
additive on the raw-output scale. SHAP contributions are therefore **not**
probability deltas.

In [7]:
explainer = shap.TreeExplainer(fitted_xgb, feature_perturbation="tree_path_dependent")
shap_values = explainer.shap_values(X_train_transformed.values)

if isinstance(shap_values, list):
    # Legacy API may return one array per class; keep the positive class.
    shap_values = shap_values[1]
shap_values = np.asarray(shap_values)
base_value = float(explainer.expected_value)

print("SHAP values shape:", shap_values.shape)
print("Base value (expected margin on training data):", round(base_value, 4))
print("Baseline probability (sigmoid(base)):", round(1 / (1 + np.exp(-base_value)), 4))
print("SHAP values are on the raw-output (log-odds) scale.")

SHAP values shape: (4907, 45)
Base value (expected margin on training data): 0.0225
Baseline probability (sigmoid(base)): 0.5056
SHAP values are on the raw-output (log-odds) scale.


## 7. Global SHAP Importance

**Global feature importance** is defined here as the **mean absolute SHAP
value** of each transformed feature across the explanation dataset (the
training set, 4,907 customers):

$$\text{mean}|\phi_j| = \frac{1}{n}\sum_{i=1}^{n} |\phi_{j}^{(i)}|$$

This measures the *average magnitude* of the feature's contribution to the
model's raw output across observations. It is **not**:
- a model coefficient (logistic regression coefficients are per-feature and
  assume linearity),
- XGBoost feature gain (a training-time split-based measure),
- permutation importance (a drop-in-performance measure).

Larger mean |SHAP| = greater average influence on the model's output — not
causal importance.

In [8]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
global_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap,
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
global_importance["rank"] = global_importance.index + 1

print(f"Global SHAP importance across {len(X_train_transformed)} training observations "
      f"({len(feature_names)} transformed features):")
display(global_importance.head(20).round(4))

Global SHAP importance across 4907 training observations (45 transformed features):


,feature,mean_abs_shap,rank
0,categorical__Contract_Month-to-month,0.6318,1
1,numerical__tenure,0.3952,2
2,numerical__MonthlyCharges,0.2934,3
3,categorical__InternetService_Fiber optic,0.2305,4
4,categorical__OnlineSecurity_No,0.2009,5
5,numerical__TotalCharges,0.1953,6
6,categorical__PaymentMethod_Electronic check,0.1811,7
7,categorical__TechSupport_No,0.1654,8
8,categorical__Contract_Two year,0.1621,9
9,categorical__PaperlessBilling_No,0.1541,10


## 8. Global SHAP Importance — Bar Plot

Top 20 transformed features, sorted by mean |SHAP|. Larger values indicate a
larger average influence on the model's output across the training data, not
causal importance.

In [9]:
top_n = 20
plot_df = global_importance.head(top_n).iloc[::-1]  # ascending for horizontal bars

fig, ax = plt.subplots(figsize=(9, 8.5))
ax.barh(plot_df["feature"], plot_df["mean_abs_shap"], color="#1f77b4")
ax.set_xlabel("Mean |SHAP value|")
ax.set_ylabel("Transformed feature")
ax.set_title(f"Global SHAP Feature Importance (top {top_n}, training set)")
plt.tight_layout()
plt.show()

## 9. SHAP Summary (Beeswarm) Plot

Each point is one customer; rows are features (top to bottom by mean |SHAP|).

- **Horizontal position** is the customer's SHAP value for that feature:
  positive = pushed that customer *toward churn*, negative = *away from churn`.
- **Colour** is the feature's value for that customer (red = high value,
  blue = low value). For one-hot encoded categorical features the value is
  binary: 1 = the category is present, 0 = it is absent. "High/low" is only a
  meaningful description for the scaled numerical features (e.g. `tenure`).
- **Spread** shows how variable that feature's contribution is across
  customers.

In [10]:
explanation = shap.Explanation(
    values=shap_values,
    base_values=np.full(shap_values.shape[0], base_value),
    data=X_train_transformed.values,
    feature_names=feature_names,
)

shap.plots.beeswarm(explanation, max_display=20, show=False)
plt.tight_layout()
plt.show()

## 10. SHAP Dependence Plots (Top 3 Features)

Dependence plots show, for one transformed feature, how the feature's value
relates to its SHAP contribution *within the fitted model*. The top 3 features
are selected **programmatically** from the computed global ranking (not
hard-coded). No interaction colouring is used, to avoid over-interpreting
correlations.

For the one-hot encoded feature in the top 3, the x-axis is binary (0 = absent,
1 = present); the plot simply shows the distribution of that category's
contribution in each case.

In [11]:
top3 = global_importance.head(3)["feature"].tolist()
print("Top 3 features for dependence plots:", top3)

for feat in top3:
    idx = int(np.where(feature_names == feat)[0][0])
    shap.dependence_plot(
        idx, shap_values, X_train_transformed.values,
        feature_names=feature_names, interaction_index=None, show=False,
    )
    plt.tight_layout()
    plt.show()

Top 3 features for dependence plots: ['categorical__Contract_Month-to-month', 'numerical__tenure', 'numerical__MonthlyCharges']


## Global Interpretation

The following is read directly from the computed global SHAP ranking (section 7),
which is based on the fitted XGBoost model and the 4,907 training observations.
Numbers are mean |SHAP| values on the raw-output (log-odds) scale.

According to the fitted XGBoost model, the transformed features with the
largest average SHAP magnitude were:

| Rank | Transformed feature | Mean |SHAP| |
|---|---|---|---|
| 1 | `categorical__Contract_Month-to-month` | 0.6318 |
| 2 | `numerical__tenure` | 0.3952 |
| 3 | `numerical__MonthlyCharges` | 0.2934 |
| 4 | `categorical__InternetService_Fiber optic` | 0.2305 |
| 5 | `categorical__OnlineSecurity_No` | 0.2009 |
| 6 | `numerical__TotalCharges` | 0.1953 |
| 7 | `categorical__PaymentMethod_Electronic check` | 0.1811 |
| 8 | `categorical__TechSupport_No` | 0.1654 |
| 9 | `categorical__Contract_Two year` | 0.1621 |
| 10 | `categorical__PaperlessBilling_No` | 0.1541 |

Interpretation (cautious, model-attribution language):

- The model's output moved the most, on average, in response to whether the
  customer was on a **month-to-month contract** (mean |SHAP| 0.6318), followed
  by **tenure** (0.3952) and **MonthlyCharges** (0.2934). According to the
  fitted model, these features had relatively large average SHAP magnitude
  across training observations.
- Several one-hot encoded signals appear next: **Fiber optic** internet
  service, **no OnlineSecurity**, **electronic check** payment, **no
  TechSupport**, and **no paperless billing**.
- These findings describe the *fitted model's* behaviour. They do **not**
  establish that any feature causes churn.

## 12. Local (Per-Customer) Explanation Function

`explain_customer(customer_index)`:

1. takes one customer (by row index in the modeling data `X`),
2. applies the **exact same preprocessing** as the final XGBoost model,
3. generates the model's predicted churn probability,
4. derives the predicted class using the fixed threshold **0.525**,
5. computes that customer's SHAP values with the same fitted explainer,
6. ranks features by absolute SHAP magnitude,
7. returns a structured explanation object.

`direction` is derived from the sign of the SHAP value:

- `shap_value > 0` → **toward churn** (pushed the model's output higher)
- `shap_value < 0` → **away from churn**

No direction is assigned manually.

In [12]:
def decode_feature(feature_name):
    '''Split a transformed feature name into (kind, original_column[, category]).'''
    if feature_name.startswith("numerical__"):
        return ("numerical", feature_name.split("__", 1)[1], None)
    if feature_name.startswith("categorical__"):
        _, rest = feature_name.split("__", 1)
        col, _, category = rest.rpartition("_")
        return ("categorical", col, category)
    return ("unknown", feature_name, None)


def explain_customer(customer_index):
    '''Return a structured SHAP explanation for one customer.

    The returned object is the evidence payload consumed by Notebook 05:
    {
      "customer_index": int,
      "predicted_probability": float,
      "threshold": float,
      "predicted_class": int,
      "base_value": float,
      "evidence": DataFrame[feature, original_feature, value,
                            value_display, shap_value, abs_shap, direction]
    }
    '''
    row = X.loc[[customer_index]]
    X_row = fitted_preprocessor.transform(row)

    proba = float(fitted_xgb.predict_proba(X_row)[0, 1])
    predicted_class = int(proba >= DECISION_THRESHOLD)
    row_shap = np.asarray(explainer.shap_values(X_row))[0]
    original_row = df_model.loc[customer_index]

    evidence = []
    for i, fname in enumerate(feature_names):
        kind, col, category = decode_feature(fname)
        value = float(X_row[0, i])
        if kind == "numerical":
            value_display = str(original_row[col])
        elif kind == "categorical":
            value_display = category if value == 1.0 else f"not {category}"
        else:
            value_display = str(value)
        direction = "toward churn" if row_shap[i] >= 0 else "away from churn"
        evidence.append({
            "feature": fname,
            "original_feature": col,
            "value": value,
            "value_display": value_display,
            "shap_value": float(row_shap[i]),
            "direction": direction,
        })

    evidence_df = pd.DataFrame(evidence).assign(
        abs_shap=lambda d: d["shap_value"].abs()
    ).sort_values("abs_shap", ascending=False).reset_index(drop=True)

    return {
        "customer_index": int(customer_index),
        "predicted_probability": proba,
        "threshold": float(DECISION_THRESHOLD),
        "predicted_class": predicted_class,
        "base_value": base_value,
        "evidence": evidence_df,
    }

## 13. Local Waterfall Plot — Example Customer

An example customer is selected programmatically from the **holdout set** (a
customer the model predicts as churning with high confidence). This customer
is being **explained**, not used to retrain or tune the model — the holdout
remains evaluation-only.

The waterfall shows the base value (0.0225 on the log-odds scale), how each
feature pushed the margin up (toward churn, red) or down (away from churn,
blue), and the final margin whose sigmoid is the predicted probability.

In [13]:
proba_holdout = fitted_xgb.predict_proba(
    fitted_preprocessor.transform(X_holdout)
)[:, 1]
tp_mask = (
    (y_holdout.values == 1)
    & (proba_holdout >= DECISION_THRESHOLD)
    & (proba_holdout >= 0.70)
    & (proba_holdout <= 0.95)
)
example_index = int(X_holdout.index[tp_mask][0])
print("Selected example customer (holdout index):", example_index)
print("Actual outcome (holdout label):",
      "Churned" if int(y_holdout.loc[example_index]) == 1 else "Retained")

example = explain_customer(example_index)
print(f"Predicted probability of churn: {example['predicted_probability']:.4f}")
print(f"Decision threshold:             {example['threshold']:.3f}")
print(f"Predicted class:                "
      f"{'Churn' if example['predicted_class'] == 1 else 'Retained'}")

Selected example customer (holdout index): 38
Actual outcome (holdout label): Churned
Predicted probability of churn: 0.7301
Decision threshold:             0.525
Predicted class:                Churn


In [14]:
single = shap.Explanation(
    values=example["evidence"]["shap_value"].values,
    base_values=base_value,
    data=example["evidence"]["value"].values,
    feature_names=example["evidence"]["feature"].values,
)
shap.plots.waterfall(single, max_display=15, show=False)
plt.tight_layout()
plt.show()

## 14. Local Explanation Table

The same customer's contributions, sorted by absolute SHAP value. Only
computed values are shown. `value_display` shows the human-readable original
value (for one-hot features, the category when present).

In [15]:
display(
    example["evidence"].head(12)[
        ["feature", "value_display", "shap_value", "direction"]
    ].round(4)
)

,feature,value_display,shap_value,direction
0,numerical__MonthlyCharges,106.35,0.5016,toward churn
1,numerical__tenure,34,-0.4771,away from churn
2,numerical__TotalCharges,3549.25,-0.4243,away from churn
3,categorical__Contract_Month-to-month,Month-to-month,0.3872,toward churn
4,categorical__InternetService_Fiber optic,Fiber optic,0.2191,toward churn
5,categorical__PaymentMethod_Electronic check,Electronic check,0.1986,toward churn
6,categorical__PaperlessBilling_No,not No,0.1314,toward churn
7,categorical__TechSupport_No,No,0.1195,toward churn
8,categorical__OnlineSecurity_No,No,0.1156,toward churn
9,categorical__StreamingTV_Yes,Yes,0.1018,toward churn


### Result (computed from data)

For holdout customer **index 38** (predicted probability **0.7301** → predicted
**Churn** at threshold 0.525; the customer did in fact churn), the model's
output was pushed toward churn mainly by:

- **MonthlyCharges** (SHAP +0.5016) — high monthly charges
- **Contract = Month-to-month** (SHAP +0.3872)
- **InternetService = Fiber optic** (SHAP +0.2191)
- **PaymentMethod = Electronic check** (SHAP +0.1986)

and pushed away from churn mainly by:

- **tenure** (SHAP -0.4771) — 34 months
- **TotalCharges** (SHAP -0.4243)

These numbers say the listed features *pushed the model toward/away from a
higher predicted churn risk* for this customer. They do not say any feature
"caused" the customer to churn.

## 15. Customer Explanation API (evidence for Notebook 05)

`explain_customer` returns a reusable, JSON-serialisable structure. Notebook 05
will consume this exact shape to generate natural-language explanations. Each
`evidence` entry is one feature:

```
{
  "customer_index": int,
  "predicted_probability": float,
  "threshold": 0.525,
  "predicted_class": int (1 = churn),
  "base_value": float,
  "evidence": [
    {"feature", "original_feature", "value", "value_display",
     "shap_value", "direction"}, ...
  ]
}
```

The example evidence is also written to a JSON file so Notebook 05 can load it
without re-running the whole model.

In [16]:
example_dict = {
    "customer_index": example["customer_index"],
    "predicted_probability": example["predicted_probability"],
    "threshold": example["threshold"],
    "predicted_class": example["predicted_class"],
    "base_value": example["base_value"],
    "evidence": example["evidence"].to_dict("records"),
}

import json

EVIDENCE_DIR = PROJECT_ROOT / "data" / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
EVIDENCE_PATH = EVIDENCE_DIR / "example_customer_evidence.json"
with open(EVIDENCE_PATH, "w") as f:
    json.dump(example_dict, f, indent=2)
print("Evidence written to:", EVIDENCE_PATH)
print("Top 3 evidence entries (feature | value_display | shap_value | direction):")
for e in example_dict["evidence"][:3]:
    print(f"  {e['feature']:<45s} {str(e['value_display']):<20s} "
          f"{e['shap_value']:+.4f}  {e['direction']}")

Evidence written to: /Users/vaibhavvikasranjan/Downloads/telco-churn-analysis/data/evidence/example_customer_evidence.json
Top 3 evidence entries (feature | value_display | shap_value | direction):
  numerical__MonthlyCharges                     106.35               +0.5016  toward churn
  numerical__tenure                             34                   -0.4771  away from churn
  numerical__TotalCharges                       3549.25              -0.4243  away from churn


## 16. Evidence Integrity Checks

Programmatic assertions that every SHAP value, feature name, prediction
probability, and threshold used above is real, aligned, and produced by the
same fitted model.

In [17]:
import numpy as np

print("Running evidence integrity checks ...")

# 1. SHAP dimension equals the transformed feature dimension.
assert shap_values.shape[1] == X_train_transformed.shape[1] == len(feature_names)
print("[ok] shap_values.shape[1] == X_train_transformed.shape[1] == len(feature_names)")

# 2. Every SHAP value is numeric and finite.
assert np.issubdtype(shap_values.dtype, np.number)
assert np.all(np.isfinite(shap_values))
print("[ok] all SHAP values are numeric and finite")

# 3. Local explanation: one entry per transformed feature, arrays match length.
evidence_df = example["evidence"]
assert len(evidence_df) == len(feature_names)
assert set(evidence_df["feature"]) == set(feature_names)
print("[ok] local evidence covers exactly the transformed feature set")

# 4. Prediction probability is generated by the fitted model.
proba_recomputed = float(fitted_xgb.predict_proba(
    fitted_preprocessor.transform(X.loc[[example_index]])
)[0, 1])
assert abs(proba_recomputed - example["predicted_probability"]) < 1e-9
print("[ok] predicted_probability recomputed from the fitted model")

# 5. Threshold matches the metadata value (0.525), compared with tolerance.
assert example["threshold"] == DECISION_THRESHOLD
assert abs(DECISION_THRESHOLD - 0.525) < 1e-6
print("[ok] threshold ==", DECISION_THRESHOLD, "(metadata; ~0.525)")

# 6. Global SHAP additivity: sum(SHAP) + base == margin (raw output).
margin_all = fitted_xgb.get_booster().predict(
    xgboost.DMatrix(X_train_transformed.values.astype(np.float64)),
    output_margin=True,
)
recon = shap_values.sum(axis=1) + base_value
max_abs_err = float(np.abs(recon - margin_all).max())
print(f"[ok] max |sum(SHAP) + base - margin| = {max_abs_err:.2e}")
assert max_abs_err < 1e-3

# 7. Local additivity for the example customer.
row_margin = float(fitted_xgb.get_booster().predict(
    xgboost.DMatrix(fitted_preprocessor.transform(X.loc[[example_index]]).astype(np.float64)),
    output_margin=True,
)[0])
local_recon = evidence_df["shap_value"].sum() + base_value
assert abs(local_recon - row_margin) < 1e-3
assert abs(example["predicted_probability"] - 1 / (1 + np.exp(-row_margin))) < 1e-6
print(f"[ok] local additivity holds (reconstructed margin {local_recon:.4f} vs {row_margin:.4f})")
print(f"[ok] sigmoid(margin) == predicted probability")

print("All evidence integrity checks passed.")

Running evidence integrity checks ...
[ok] shap_values.shape[1] == X_train_transformed.shape[1] == len(feature_names)
[ok] all SHAP values are numeric and finite
[ok] local evidence covers exactly the transformed feature set
[ok] predicted_probability recomputed from the fitted model
[ok] threshold == 0.525 (metadata; ~0.525)
[ok] max |sum(SHAP) + base - margin| = 2.38e-06
[ok] local additivity holds (reconstructed margin 0.9949 vs 0.9949)
[ok] sigmoid(margin) == predicted probability
All evidence integrity checks passed.


## 17. Global vs Local Explanation

- **GLOBAL:** *Which features generally influence the model's predictions across
  the explanation dataset?* Answered by mean |SHAP| (section 7). It summarises
  average behaviour, not any single customer.

- **LOCAL:** *Why did the model assign this particular customer this particular
  prediction?* Answered by one customer's SHAP vector (sections 13–14). It is
  specific to that observation's feature values.

A feature can be **globally important but locally weak** (e.g. its average
contribution is large, but for this customer its value happens to produce a
small contribution), and **globally weak but locally decisive** (rarely
important on average, yet the deciding factor for this customer). Both views
are necessary for a full explanation.

## 18. Important Statistical / Modeling Distinction

> **SHAP values describe the behaviour of the fitted predictive model. They do
> not demonstrate that a feature causes customer churn.**

- **Notebook 02** — *statistical association*: whether observed relationships
  in the dataset are unlikely under the null hypothesis (chi-square / Mann-
  Whitney U).
- **Notebook 03** — *predictive performance*: how well the model generalises to
  unseen (holdout) data.
- **Notebook 04** — *model attribution*: which features, and how strongly,
  drove the fitted model's outputs.

SHAP can tell us the model leans on month-to-month contracts; it cannot tell us
that month-to-month contracts *cause* churn. This keeps the project's
scientific claims honest.

## 19. Limitations

Honest limits of this explainability analysis:

1. **SHAP explains the fitted model, not causal relationships.** SHAP
   attributes the model's output; it does not identify causes of churn in the
   real world.
2. **One-hot encoding reduces categorical interpretability.** Each category is
   its own transformed feature; "the model uses Contract" must be read through
   its per-category one-hot columns.
3. **Correlated features can split attribution.** When features are correlated
   (e.g. `tenure`, `TotalCharges`, `MonthlyCharges`), SHAP can distribute
   credit among them, so individual values should not be over-read in
   isolation.
4. **SHAP importance depends on the model and the explanation dataset.** Mean
   |SHAP| here is computed on the training set for the Notebook 03 XGBoost
   configuration; a different model or dataset would give different numbers.
5. **Local explanations describe model behaviour for one observation.** They
   are not a guarantee that the model's prediction for that customer is
   correct.
6. **Dataset scope.** The cleaned dataset has 7,032 rows; Notebook 03 used
   7,010 rows for modeling after removing 22 exact duplicate rows. This
   notebook follows the same modeling copy.
7. **Generalization.** Explanations may not generalize to other telecom
   populations, markets, pricing structures, or time periods.
8. **Model correctness.** SHAP faithfully explains what the model did; it says
   nothing about whether the model's prediction is factually right for any
   customer.

No claim is made beyond what the executed code produced.